In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/Lung_Colon/colon_image_sets/ /content/

In [ ]:
!cp -r /content/drive/MyDrive/ViT /content/

In [ ]:
!pip install open_clip_torch

In [ ]:
!pip install -q timm transformers nibabel open_clip_torch segment-anything huggingface_hub

In [ ]:
# ===================================== 2D IMAGE RETRIEVAL BENCHMARK =====================================
# Folder structure:
#   dataset_root/
#       class_a/
#           image1.jpg
#           image2.png
#       class_b/
#           image3.jpg
#           ...
#
# What it does:
#   - Stratified 80/20 split by class
#   - Extract embeddings for both train and test splits
#   - Retrieval: for each test image, retrieve top-k most similar train images
#   - Evaluate whether retrieved neighbors have the same class as the query
#
# Metrics:
#   - Precision@K
#   - HitRate@K
#   - Recall@K
#   - mAP
#   - MRR
#
# Added models:
#   - MedSigLIP
#   - SigLIP
#   - UNI (medical foundation model; requires access if gated)
#   - OpenCLIP ViT-H/14
# ========================================================================================================

import os
import gc
import random
from pathlib import Path
from typing import List, Dict, Optional

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Install if needed:
# !pip install -q timm transformers open_clip_torch segment-anything huggingface_hub pillow scikit-learn

import timm
from sklearn.model_selection import train_test_split
from transformers import CLIPVisionModel, SamModel, AutoModel
from huggingface_hub import hf_hub_download

try:
    import open_clip
    HAS_OPEN_CLIP = True
except Exception:
    HAS_OPEN_CLIP = False

try:
    from segment_anything import sam_model_registry
    HAS_SEGMENT_ANYTHING = True
except Exception:
    HAS_SEGMENT_ANYTHING = False


# ===================================== USER CONFIG =====================================
DATASET_ROOT = "/content/colon_image_sets"
RESULTS_DIR = "./retrieval_outputs_2d"
os.makedirs(RESULTS_DIR, exist_ok=True)

BACKBONES_TO_TEST = [
    "resnet50",
    "vit_b16",
    "swin_b",
    "clip_vit_l14",
    "biomedclip",
    "dinov2_vitb14",
    "mae_vit_b16",
    "bmc_clip_cf",
    "sam_vit_b",
    "medsam_vit_b",
    "siglip",
    "medsiglip",
    "uni",
    "openclip_vit_h14",
]

MODEL_PATHS = {
    "clip_vit_l14": "openai/clip-vit-large-patch14",
    "biomedclip": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    "sam_vit_b": "facebook/sam-vit-base",

    # Put your local MedSAM checkpoint here
    "medsam_vit_b_ckpt": "/content/ViT/medsam_vit_b.pth",

    # BMC-CLIP-CF
    "bmc_clip_cf_repo": "BIOMEDICA/BMC_CLIP_CF",
    "bmc_clip_cf_file": "BMC_CLIP_CF.pt",
    "bmc_clip_cf_base_model": "ViT-L-14",
    "bmc_clip_cf_base_pretrained": "commonpool_xl_clip_s13b_b90k",
    "bmc_clip_cf_alpha": 0.0,

    # Added
    "siglip": "google/siglip-base-patch16-224",
    "medsiglip": "google/medsiglip-448",

    # UNI can be gated; ensure access if needed
    "uni": "hf-hub:MahmoodLab/UNI",

    # OpenCLIP ViT-H/14
    "openclip_vit_h14_model": "ViT-H-14",
    "openclip_vit_h14_pretrained": "laion2b_s32b_b79k",
}

IMAGE_SIZE = 224
TOP_KS = (1, 5, 10)
BATCH_SIZE = 32
NUM_WORKERS = 2
EMBED_DIM = 512
USE_BFLOAT16 = True
RANDOM_SEED = 42
SHOW_RESULTS = True
NUM_EXAMPLE_QUERIES = 5

ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
# =======================================================================================


# ===================================== NORMALIZATION =====================================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

CLIP_MEAN = torch.tensor([0.48145466, 0.45782750, 0.40821073], dtype=torch.float32).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], dtype=torch.float32).view(1, 3, 1, 1)


def imagenet_norm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(device=x.device, dtype=x.dtype)
    std = IMAGENET_STD.to(device=x.device, dtype=x.dtype)
    return (x - mean) / std


def clip_norm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = CLIP_MEAN.to(device=x.device, dtype=x.dtype)
    std = CLIP_STD.to(device=x.device, dtype=x.dtype)
    return (x - mean) / std


def imagenet_unnorm_batch(x: torch.Tensor) -> torch.Tensor:
    mean = IMAGENET_MEAN.to(device=x.device, dtype=x.dtype)
    std = IMAGENET_STD.to(device=x.device, dtype=x.dtype)
    return x * std + mean


# ===================================== UTILS =====================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_open_rgb(path: str) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        raise RuntimeError(f"Could not open image {path}: {e}")


def list_images_by_class(root: str) -> pd.DataFrame:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {root}")

    rows = []
    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])

    if len(class_dirs) == 0:
        raise ValueError(f"No class folders found in {root}")

    for class_dir in class_dirs:
        class_name = class_dir.name
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in ALLOWED_EXTS:
                rows.append({
                    "image_path": str(p),
                    "class_name": class_name,
                })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise ValueError(f"No images found in {root}")

    classes = sorted(df["class_name"].unique().tolist())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    df["label"] = df["class_name"].map(class_to_idx)
    return df


def stratified_split(df: pd.DataFrame, test_size: float = 0.2, seed: int = 42):
    counts = df["label"].value_counts()
    valid_labels = counts[counts >= 2].index.tolist()
    dropped = df[~df["label"].isin(valid_labels)].copy()
    df = df[df["label"].isin(valid_labels)].copy().reset_index(drop=True)

    if len(dropped) > 0:
        print(f"[WARN] Dropped {len(dropped)} images from classes with <2 images")

    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"].values,
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def average_precision_from_flags(relevant_flags: List[int]) -> float:
    relevant_flags = np.asarray(relevant_flags, dtype=np.int32)
    total_rel = relevant_flags.sum()
    if total_rel == 0:
        return 0.0

    precisions = []
    hits = 0
    for i, rel in enumerate(relevant_flags, start=1):
        if rel:
            hits += 1
            precisions.append(hits / i)
    return float(np.mean(precisions)) if precisions else 0.0


def reciprocal_rank(relevant_flags: List[int]) -> float:
    for i, rel in enumerate(relevant_flags, start=1):
        if rel:
            return 1.0 / i
    return 0.0


# ===================================== DATASET =====================================
class ImageFolderRetrievalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def _preprocess(self, img: Image.Image) -> torch.Tensor:
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        x = np.asarray(img, dtype=np.float32) / 255.0
        x = np.transpose(x, (2, 0, 1))  # HWC -> CHW
        x = torch.from_numpy(x)

        # Store dataset output as ImageNet-normalized, then convert inside retriever
        mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
        x = (x - mean) / std
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_rgb(row["image_path"])
        x = self._preprocess(img)
        return {
            "image": x,
            "label": int(row["label"]),
            "class_name": row["class_name"],
            "image_path": row["image_path"],
        }


# ===================================== BMC-CLIP-CF LOADER =====================================
def load_bmc_clip_cf():
    if not HAS_OPEN_CLIP:
        raise ImportError("BMC_CLIP_CF requires open_clip_torch.")

    repo_id = MODEL_PATHS["bmc_clip_cf_repo"]
    filename = MODEL_PATHS["bmc_clip_cf_file"]
    model_name = MODEL_PATHS["bmc_clip_cf_base_model"]
    pretrained = MODEL_PATHS["bmc_clip_cf_base_pretrained"]
    alpha = float(MODEL_PATHS.get("bmc_clip_cf_alpha", 0.0))

    ckpt_path = hf_hub_download(repo_id=repo_id, filename=filename)
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint["state_dict"]
    del checkpoint

    model, _, _ = open_clip.create_model_and_transforms(
        model_name=model_name,
        pretrained=pretrained,
    )

    model_state_dict = model.state_dict()

    print(f"Merging BMC-CLIP-CF with alpha={alpha}")
    if alpha == 0:
        print("Using only BMC-CLIP-CF checkpoint weights")

    for key in model_state_dict.keys():
        ckpt_key = f"module.{key}"
        if ckpt_key in state_dict:
            model_state_dict[key] = alpha * model_state_dict[key] + (1 - alpha) * state_dict[ckpt_key]

    model.load_state_dict(model_state_dict, strict=True)
    return model


# ===================================== FEATURE EXTRACTORS =====================================
def create_feature_extractor(backbone_name: str):
    """
    Returns:
      model, feat_dim, encoder_kind, preferred_image_size
    """
    backbone_name = backbone_name.lower()

    if backbone_name == "resnet50":
        m = timm.create_model("resnet50", pretrained=True, num_classes=0)
        return m, m.num_features, "timm_slice2d", 224

    elif backbone_name == "vit_b16":
        m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
        return m, m.embed_dim, "timm_slice2d", 224

    elif backbone_name == "swin_b":
        m = timm.create_model("swin_base_patch4_window7_224", pretrained=True, num_classes=0)
        return m, m.num_features, "timm_slice2d", 224

    elif backbone_name == "clip_vit_l14":
        m = CLIPVisionModel.from_pretrained(MODEL_PATHS["clip_vit_l14"])
        return m, m.config.hidden_size, "clip_slice2d", 224

    elif backbone_name == "biomedclip":
        if not HAS_OPEN_CLIP:
            raise ImportError("BiomedCLIP requires open_clip_torch.")
        m, _, _ = open_clip.create_model_and_transforms(MODEL_PATHS["biomedclip"])
        feat_dim = getattr(m.visual, "output_dim", 512)
        return m, feat_dim, "openclip_slice2d", 224

    elif backbone_name == "dinov2_vitb14":
        m = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        return m, m.embed_dim, "dinov2_slice2d", 224

    elif backbone_name == "mae_vit_b16":
        m = timm.create_model("vit_base_patch16_224.mae", pretrained=True, num_classes=0)
        return m, m.embed_dim, "timm_slice2d", 224

    elif backbone_name == "bmc_clip_cf":
        m = load_bmc_clip_cf()
        feat_dim = getattr(m.visual, "output_dim", 768)
        return m, feat_dim, "openclip_slice2d", 224

    elif backbone_name == "sam_vit_b":
        m = SamModel.from_pretrained(MODEL_PATHS["sam_vit_b"])
        feat_dim = int(getattr(m.config.vision_config, "output_channels", 256))
        return m, feat_dim, "sam_hf_slice2d", 1024

    elif backbone_name == "medsam_vit_b":
        if not HAS_SEGMENT_ANYTHING:
            raise ImportError("MedSAM requires segment-anything.")
        ckpt = MODEL_PATHS["medsam_vit_b_ckpt"]
        if not os.path.isfile(ckpt):
            raise FileNotFoundError(f"MedSAM checkpoint not found: {ckpt}")
        m = sam_model_registry["vit_b"](checkpoint=ckpt)
        return m, 256, "sam_repo_slice2d", 512

    elif backbone_name == "siglip":
        m = AutoModel.from_pretrained(MODEL_PATHS["siglip"])
        hidden = getattr(getattr(m, "config", None), "hidden_size", None)
        if hidden is None and hasattr(m, "vision_model") and hasattr(m.vision_model.config, "hidden_size"):
            hidden = m.vision_model.config.hidden_size
        if hidden is None:
            hidden = 768
        return m, hidden, "siglip_slice2d", 224

    elif backbone_name == "medsiglip":
        m = AutoModel.from_pretrained(MODEL_PATHS["medsiglip"])
        hidden = getattr(getattr(m, "config", None), "hidden_size", None)
        if hidden is None and hasattr(m, "vision_model") and hasattr(m.vision_model.config, "hidden_size"):
            hidden = m.vision_model.config.hidden_size
        if hidden is None:
            hidden = 768
        return m, hidden, "medsiglip_slice2d", 448

    elif backbone_name == "uni":
        try:
            m = timm.create_model(MODEL_PATHS["uni"], pretrained=True, num_classes=0)
        except Exception:
            # Some releases/cards recommend these extra args
            m = timm.create_model(
                MODEL_PATHS["uni"],
                pretrained=True,
                num_classes=0,
                init_values=1e-5,
                dynamic_img_size=True,
            )

        feat_dim = getattr(m, "num_features", None)
        if feat_dim is None:
            feat_dim = getattr(m, "embed_dim", 1024)
        return m, feat_dim, "uni_slice2d", 224

    elif backbone_name == "openclip_vit_h14":
        if not HAS_OPEN_CLIP:
            raise ImportError("openclip_vit_h14 requires open_clip_torch.")
        m, _, _ = open_clip.create_model_and_transforms(
            model_name=MODEL_PATHS["openclip_vit_h14_model"],
            pretrained=MODEL_PATHS["openclip_vit_h14_pretrained"],
        )
        feat_dim = getattr(m.visual, "output_dim", 1024)
        return m, feat_dim, "openclip_slice2d", 224

    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")


# ===================================== RETRIEVER =====================================
class UniversalImageRetriever(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        embed_dim: Optional[int] = 512,
        use_bfloat16: bool = True,
        image_size: int = 224,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.encoder, self.hidden_size, self.encoder_kind, self.preferred_image_size = create_feature_extractor(self.backbone_name)

        self.image_size = self.preferred_image_size if self.preferred_image_size is not None else image_size

        self.proj = None
        self.output_dim = self.hidden_size
        if embed_dim is not None and embed_dim != self.hidden_size:
            self.proj = nn.Linear(self.hidden_size, embed_dim)
            self.output_dim = embed_dim

        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()

        safe_bf16_kinds = {
            "timm_slice2d",
            "clip_slice2d",
            "openclip_slice2d",
            "dinov2_slice2d",
            "uni_slice2d",
            "siglip_slice2d",
            "medsiglip_slice2d",
        }
        if use_bfloat16 and torch.cuda.is_available() and self.encoder_kind in safe_bf16_kinds:
            self.encoder = self.encoder.to(dtype=torch.bfloat16)

    def _adapt_input_for_backbone(self, x: torch.Tensor) -> torch.Tensor:
        """
        Dataset outputs ImageNet-normalized tensors.
        Convert back to [0,1], then apply the right normalization per backbone.
        """
        x = imagenet_unnorm_batch(x.float()).clamp(0.0, 1.0)

        if self.encoder_kind in {"clip_slice2d", "openclip_slice2d", "siglip_slice2d", "medsiglip_slice2d"}:
            x = clip_norm_batch(x)
        elif self.encoder_kind in {"timm_slice2d", "dinov2_slice2d", "sam_hf_slice2d", "sam_repo_slice2d", "uni_slice2d"}:
            x = imagenet_norm_batch(x)
        else:
            x = imagenet_norm_batch(x)

        return x

    def _normalize_encoder_output(self, feats):
        if isinstance(feats, dict):
            for key in [
                "image_embeds",
                "x_norm_clstoken",
                "x_cls_token",
                "cls_token",
                "pooler_output",
                "last_hidden_state",
                "x_norm_patchtokens",
            ]:
                if key in feats:
                    feats = feats[key]
                    break
            else:
                first_key = list(feats.keys())[0]
                feats = feats[first_key]

        if isinstance(feats, (list, tuple)):
            feats = feats[0]

        if hasattr(feats, "last_hidden_state"):
            feats = feats.last_hidden_state

        if not torch.is_tensor(feats):
            raise TypeError(f"Unsupported feature output type: {type(feats)}")

        if feats.ndim == 4:
            feats = feats.mean(dim=(2, 3))
        elif feats.ndim == 3:
            feats = feats.mean(dim=1)
        elif feats.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected feature tensor shape: {tuple(feats.shape)}")

        return feats

    def _forward_in_chunks(self, x: torch.Tensor, fn, chunk_size: int = 2) -> torch.Tensor:
        outs = []
        for i in range(0, x.shape[0], chunk_size):
            xi = x[i:i + chunk_size]
            yi = fn(xi)
            outs.append(yi)
        return torch.cat(outs, dim=0)

    def _encode_2d_batch(self, x2d: torch.Tensor) -> torch.Tensor:
        if self.encoder_kind == "timm_slice2d":
            if self.backbone_name == "swin_b":
                feats = self.encoder.forward_features(x2d)

                if isinstance(feats, (list, tuple)):
                    feats = feats[0]

                if not torch.is_tensor(feats):
                    raise TypeError(f"Unsupported Swin output type: {type(feats)}")

                if feats.ndim == 4:
                    if feats.shape[-1] == self.hidden_size:
                        feats = feats.mean(dim=(1, 2))   # NHWC
                    elif feats.shape[1] == self.hidden_size:
                        feats = feats.mean(dim=(2, 3))   # NCHW
                    else:
                        raise ValueError(f"Unexpected Swin feature shape: {tuple(feats.shape)}")
                elif feats.ndim == 3:
                    feats = feats.mean(dim=1)
                elif feats.ndim == 2:
                    pass
                else:
                    raise ValueError(f"Unexpected Swin feature shape: {tuple(feats.shape)}")

                return feats

            else:
                if hasattr(self.encoder, "forward_features"):
                    feats = self.encoder.forward_features(x2d)
                else:
                    feats = self.encoder(x2d)
                feats = self._normalize_encoder_output(feats)
                return feats

        elif self.encoder_kind == "clip_slice2d":
            out = self.encoder(pixel_values=x2d)
            feats = out.last_hidden_state.mean(dim=1)
            return feats

        elif self.encoder_kind == "openclip_slice2d":
            if hasattr(self.encoder, "encode_image"):
                feats = self.encoder.encode_image(x2d)
            else:
                feats = self.encoder.visual(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "dinov2_slice2d":
            if hasattr(self.encoder, "forward_features"):
                feats = self.encoder.forward_features(x2d)
            else:
                feats = self.encoder(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "sam_hf_slice2d":
            x = x2d.float()
            if x.max() <= 1.0:
                x = x * 255.0

            feats = self._forward_in_chunks(
                x,
                lambda z: self.encoder.get_image_embeddings(pixel_values=z),
                chunk_size=1,
            )
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "sam_repo_slice2d":
            x = x2d.float()
            if x.max() <= 1.0:
                x = x * 255.0

            def run_repo_sam(z):
                zz = self.encoder.preprocess(z) if hasattr(self.encoder, "preprocess") else z
                return self.encoder.image_encoder(zz)

            feats = self._forward_in_chunks(x, run_repo_sam, chunk_size=1)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "siglip_slice2d":
            # Prefer direct image feature API if exposed
            if hasattr(self.encoder, "get_image_features"):
                feats = self.encoder.get_image_features(pixel_values=x2d)
            else:
                out = self.encoder(pixel_values=x2d)
                if hasattr(out, "image_embeds") and out.image_embeds is not None:
                    feats = out.image_embeds
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state.mean(dim=1)
                else:
                    feats = self._normalize_encoder_output(out)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "medsiglip_slice2d":
            if hasattr(self.encoder, "get_image_features"):
                feats = self.encoder.get_image_features(pixel_values=x2d)
            else:
                out = self.encoder(pixel_values=x2d)
                if hasattr(out, "image_embeds") and out.image_embeds is not None:
                    feats = out.image_embeds
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state.mean(dim=1)
                else:
                    feats = self._normalize_encoder_output(out)
            feats = self._normalize_encoder_output(feats)
            return feats

        elif self.encoder_kind == "uni_slice2d":
            if hasattr(self.encoder, "forward_features"):
                feats = self.encoder.forward_features(x2d)
            else:
                feats = self.encoder(x2d)
            feats = self._normalize_encoder_output(feats)
            return feats

        else:
            raise ValueError(f"Unknown encoder kind: {self.encoder_kind}")

    @torch.inference_mode()
    def encode_images(self, x: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device
        encoder_params = list(self.encoder.parameters())
        enc_dtype = encoder_params[0].dtype if len(encoder_params) > 0 else torch.float32

        # Input comes ImageNet-normalized from dataset -> adapt for backbone
        x = x.to(device=device, dtype=torch.float32)
        x = self._adapt_input_for_backbone(x)

        # Resize if needed for backbone
        if x.shape[-1] != self.image_size or x.shape[-2] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size), mode="bilinear", align_corners=False)

        x = x.to(device=device, dtype=enc_dtype)
        feats = self._encode_2d_batch(x)

        if self.proj is not None:
            feats = self.proj(feats.float())

        feats = F.normalize(feats.float(), p=2, dim=-1)
        return feats


# ===================================== EMBEDDINGS =====================================
def build_embeddings(
    df: pd.DataFrame,
    retriever: UniversalImageRetriever,
    split_name: str,
    batch_size: int = 32,
    num_workers: int = 2,
    image_size: int = 224,
):
    dataset = ImageFolderRetrievalDataset(df=df, image_size=image_size)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    rows = []
    embeds = []

    retriever.eval()

    for i, batch in enumerate(loader, start=1):
        x = batch["image"]
        labels = batch["label"].numpy()
        class_names = batch["class_name"]
        image_paths = batch["image_path"]

        feats = retriever.encode_images(x).detach().cpu().numpy().astype(np.float32)
        embeds.append(feats)

        for j in range(len(image_paths)):
            rows.append({
                "image_path": image_paths[j],
                "label": int(labels[j]),
                "class_name": class_names[j],
                "split": split_name,
            })

        if i % 10 == 0 or i == len(loader):
            print(f"[{split_name}] Processed {i}/{len(loader)} batches")

    embeddings = np.vstack(embeds).astype(np.float32)
    index_df = pd.DataFrame(rows)
    return index_df, embeddings


# ===================================== RETRIEVAL =====================================
def retrieve_topk(
    query_embeddings: np.ndarray,
    gallery_embeddings: np.ndarray,
    top_k: int = 5,
):
    query_embeddings = query_embeddings / (np.linalg.norm(query_embeddings, axis=1, keepdims=True) + 1e-8)
    gallery_embeddings = gallery_embeddings / (np.linalg.norm(gallery_embeddings, axis=1, keepdims=True) + 1e-8)

    sims = query_embeddings @ gallery_embeddings.T
    topk_idx = np.argsort(sims, axis=1)[:, ::-1][:, :top_k]
    topk_sims = np.take_along_axis(sims, topk_idx, axis=1)
    return topk_idx, topk_sims


def evaluate_retrieval(
    query_df: pd.DataFrame,
    query_embeddings: np.ndarray,
    gallery_df: pd.DataFrame,
    gallery_embeddings: np.ndarray,
    ks=(1, 5, 10),
):
    query_labels = query_df["label"].values
    gallery_labels = gallery_df["label"].values

    max_k = max(ks)
    topk_idx, topk_sims = retrieve_topk(
        query_embeddings=query_embeddings,
        gallery_embeddings=gallery_embeddings,
        top_k=max_k,
    )

    precision_at_k = {k: [] for k in ks}
    hitrate_at_k = {k: [] for k in ks}
    recall_at_k = {k: [] for k in ks}
    ap_list = []
    rr_list = []
    examples = []

    gallery_label_counts = pd.Series(gallery_labels).value_counts().to_dict()

    for q_idx in range(len(query_df)):
        q_label = int(query_labels[q_idx])
        q_class = query_df.iloc[q_idx]["class_name"]
        q_path = query_df.iloc[q_idx]["image_path"]

        retrieved_indices = topk_idx[q_idx]
        retrieved_sims = topk_sims[q_idx]
        retrieved_labels = gallery_labels[retrieved_indices]
        relevant_flags = (retrieved_labels == q_label).astype(np.int32).tolist()

        total_relevant_in_gallery = int(gallery_label_counts.get(q_label, 0))

        for k in ks:
            flags_k = relevant_flags[:k]
            num_rel_k = sum(flags_k)

            precision_at_k[k].append(num_rel_k / k)
            hitrate_at_k[k].append(1.0 if num_rel_k > 0 else 0.0)

            if total_relevant_in_gallery > 0:
                recall_at_k[k].append(num_rel_k / total_relevant_in_gallery)
            else:
                recall_at_k[k].append(0.0)

        ap_list.append(average_precision_from_flags(relevant_flags))
        rr_list.append(reciprocal_rank(relevant_flags))

        ex = {
            "query_path": q_path,
            "query_class": q_class,
            "retrieved": []
        }
        for rank, (g_idx, sim) in enumerate(zip(retrieved_indices, retrieved_sims), start=1):
            ex["retrieved"].append({
                "rank": rank,
                "image_path": gallery_df.iloc[g_idx]["image_path"],
                "class_name": gallery_df.iloc[g_idx]["class_name"],
                "label": int(gallery_df.iloc[g_idx]["label"]),
                "similarity": float(sim),
                "correct_class": bool(gallery_df.iloc[g_idx]["label"] == q_label),
            })
        examples.append(ex)

    metrics = {
        "num_queries": int(len(query_df)),
        "gallery_size": int(len(gallery_df)),
        "mAP": float(np.mean(ap_list)) if len(ap_list) > 0 else 0.0,
        "MRR": float(np.mean(rr_list)) if len(rr_list) > 0 else 0.0,
    }

    for k in ks:
        metrics[f"Precision@{k}"] = float(np.mean(precision_at_k[k])) if len(precision_at_k[k]) > 0 else 0.0
        metrics[f"HitRate@{k}"] = float(np.mean(hitrate_at_k[k])) if len(hitrate_at_k[k]) > 0 else 0.0
        metrics[f"Recall@{k}"] = float(np.mean(recall_at_k[k])) if len(recall_at_k[k]) > 0 else 0.0

    return metrics, examples


# ===================================== VIS =====================================
def show_example_retrievals(examples: List[Dict], num_examples: int = 5):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("matplotlib not available, skipping visualization.")
        return

    chosen = examples[:num_examples]
    if len(chosen) == 0:
        return

    for ex in chosen:
        retrieved = ex["retrieved"]
        ncols = len(retrieved) + 1

        plt.figure(figsize=(3 * ncols, 3))

        q_img = safe_open_rgb(ex["query_path"])
        ax = plt.subplot(1, ncols, 1)
        ax.imshow(q_img)
        ax.set_title(f"QUERY\n{ex['query_class']}")
        ax.axis("off")

        for i, item in enumerate(retrieved, start=2):
            img = safe_open_rgb(item["image_path"])
            ax = plt.subplot(1, ncols, i)
            ax.imshow(img)
            ax.set_title(
                f"R{item['rank']}\n{item['class_name']}\nsim={item['similarity']:.3f}\n"
                f"{'OK' if item['correct_class'] else 'WRONG'}",
                fontsize=9
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()


# ===================================== SAVE =====================================
def save_split_and_embeddings(split_df: pd.DataFrame, embeddings: np.ndarray, prefix: str, out_dir: str):
    csv_path = os.path.join(out_dir, f"{prefix}_index.csv")
    npy_path = os.path.join(out_dir, f"{prefix}_embeddings.npy")

    split_df.to_csv(csv_path, index=False)
    np.save(npy_path, embeddings)

    print(f"Saved {prefix} CSV: {csv_path}")
    print(f"Saved {prefix} embeddings: {npy_path}")
    print(f"{prefix} embedding shape: {embeddings.shape}")


# ===================================== MAIN =====================================
def main():
    set_seed(RANDOM_SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    df = list_images_by_class(DATASET_ROOT)
    print(f"Total images found: {len(df)}")
    print(f"Total classes found: {df['label'].nunique()}")
    print("\nClass distribution:")
    print(df["class_name"].value_counts())

    train_df, test_df = stratified_split(df, test_size=0.2, seed=RANDOM_SEED)
    print(f"\nTrain size: {len(train_df)}")
    print(f"Test size:  {len(test_df)}")

    all_metrics = []

    for backbone_name in BACKBONES_TO_TEST:
        print("\n" + "=" * 100)
        print(f"BACKBONE: {backbone_name}")
        print("=" * 100)

        try:
            retriever = UniversalImageRetriever(
                backbone_name=backbone_name,
                embed_dim=EMBED_DIM,
                use_bfloat16=USE_BFLOAT16,
                image_size=IMAGE_SIZE,
            ).to(device).eval()

            # Dataset stays simple at IMAGE_SIZE; retriever resizes internally for each backbone
            train_index_df, train_embeddings = build_embeddings(
                df=train_df,
                retriever=retriever,
                split_name="train",
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                image_size=IMAGE_SIZE,
            )

            test_index_df, test_embeddings = build_embeddings(
                df=test_df,
                retriever=retriever,
                split_name="test",
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                image_size=IMAGE_SIZE,
            )

            safe_name = backbone_name.replace("/", "_")
            out_dir = os.path.join(RESULTS_DIR, safe_name)
            os.makedirs(out_dir, exist_ok=True)

            save_split_and_embeddings(train_index_df, train_embeddings, "train", out_dir)
            save_split_and_embeddings(test_index_df, test_embeddings, "test", out_dir)

            metrics, examples = evaluate_retrieval(
                query_df=test_index_df,
                query_embeddings=test_embeddings,
                gallery_df=train_index_df,
                gallery_embeddings=train_embeddings,
                ks=TOP_KS,
            )

            metrics["backbone"] = backbone_name
            all_metrics.append(metrics)

            print("\nMetrics:")
            print(pd.DataFrame([metrics]).T)

            rows = []
            for ex in examples:
                for item in ex["retrieved"]:
                    rows.append({
                        "query_path": ex["query_path"],
                        "query_class": ex["query_class"],
                        "rank": item["rank"],
                        "retrieved_path": item["image_path"],
                        "retrieved_class": item["class_name"],
                        "similarity": item["similarity"],
                        "correct_class": item["correct_class"],
                    })
            pd.DataFrame(rows).to_csv(os.path.join(out_dir, "retrieval_examples.csv"), index=False)

            if SHOW_RESULTS:
                print(f"\nShowing example retrievals for {backbone_name}")
                show_example_retrievals(examples, num_examples=NUM_EXAMPLE_QUERIES)

            del retriever
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"[ERROR] Backbone {backbone_name} failed: {e}")

    if len(all_metrics) > 0:
        df_metrics = pd.DataFrame(all_metrics).sort_values(by="mAP", ascending=False)
        summary_csv = os.path.join(RESULTS_DIR, "retrieval_comparison_summary.csv")
        df_metrics.to_csv(summary_csv, index=False)

        print("\nFinal comparison:")
        print(df_metrics)
        print(f"\nSaved summary to: {summary_csv}")
    else:
        print("No successful backbone runs.")


if __name__ == "__main__":
    main()

In [ ]:
import os
from typing import List, Optional
import numpy as np
import pandas as pd


def load_saved_query_and_results(model_dir: str, q_idx: int):
    test_index_csv = os.path.join(model_dir, "test_index.csv")
    test_embed_npy = os.path.join(model_dir, "test_embeddings.npy")
    retrieval_csv = os.path.join(model_dir, "retrieval_examples.csv")

    if not os.path.isfile(test_index_csv):
        raise FileNotFoundError(f"Missing file: {test_index_csv}")
    if not os.path.isfile(test_embed_npy):
        raise FileNotFoundError(f"Missing file: {test_embed_npy}")
    if not os.path.isfile(retrieval_csv):
        raise FileNotFoundError(f"Missing file: {retrieval_csv}")

    test_index_df = pd.read_csv(test_index_csv)
    test_embeddings = np.load(test_embed_npy).astype(np.float32)
    retrieval_df = pd.read_csv(retrieval_csv)

    if q_idx < 0 or q_idx >= len(test_index_df):
        raise IndexError(f"q_idx={q_idx} out of range [0, {len(test_index_df)-1}]")

    query_path = test_index_df.iloc[q_idx]["image_path"]
    query_emb = test_embeddings[q_idx]

    results_for_query = (
        retrieval_df[retrieval_df["query_path"] == query_path]
        .sort_values("rank")
        .to_dict(orient="records")
    )

    if len(results_for_query) == 0:
        raise ValueError(f"No retrieval results found for query_path: {query_path}")

    return query_emb, results_for_query, query_path, test_index_df


def show_pca_with_similarity_coloring(
    query_emb: np.ndarray,
    results: List[dict],
    index_csv: str,
    embed_npy: str,
    query_path: Optional[str] = None,
    top_k: Optional[int] = None,
):
    try:
        import matplotlib.pyplot as plt
        from sklearn.decomposition import PCA
    except Exception:
        print("Please install scikit-learn and matplotlib:")
        print("!pip install -q scikit-learn matplotlib")
        return

    if not os.path.isfile(index_csv):
        raise FileNotFoundError(f"Missing file: {index_csv}")
    if not os.path.isfile(embed_npy):
        raise FileNotFoundError(f"Missing file: {embed_npy}")

    index_df = pd.read_csv(index_csv)
    db_embeddings = np.load(embed_npy).astype(np.float32)
    db_embeddings = db_embeddings / (np.linalg.norm(db_embeddings, axis=1, keepdims=True) + 1e-8)

    query_emb = np.asarray(query_emb, dtype=np.float32)
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-8)

    sims = db_embeddings @ query_emb

    if top_k is not None:
        results = results[:top_k]

    retrieved_paths = []
    for item in results:
        if "image_path" in item:
            retrieved_paths.append(item["image_path"])
        elif "retrieved_path" in item:
            retrieved_paths.append(item["retrieved_path"])
        else:
            raise KeyError("Each result must contain either 'image_path' or 'retrieved_path'.")

    db_paths_abs = [os.path.abspath(p) for p in index_df["image_path"].tolist()]
    path_to_idx = {p: i for i, p in enumerate(db_paths_abs)}

    retrieved_indices = [
        path_to_idx[os.path.abspath(p)]
        for p in retrieved_paths
        if os.path.abspath(p) in path_to_idx
    ]

    query_db_indices = []
    if query_path is not None:
        query_abs = os.path.abspath(query_path)
        query_db_indices = [i for i, p in enumerate(db_paths_abs) if p == query_abs]

    X = np.vstack([db_embeddings, query_emb[None, :]])
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)
    explained = pca.explained_variance_ratio_
    query_idx = len(X) - 1

    all_db_indices = np.arange(len(db_embeddings))
    query_db_set = set(query_db_indices)
    mask = np.array([i not in query_db_set for i in all_db_indices])

    plt.figure(figsize=(11, 8))

    sc = plt.scatter(
        X_2d[:-1][mask, 0],
        X_2d[:-1][mask, 1],
        c=sims[mask],
        s=20,
        alpha=0.55,
    )
    plt.colorbar(sc, label="Cosine similarity to query")

    cmap = sc.cmap
    norm = sc.norm

    for rank_i, db_idx in enumerate(retrieved_indices, start=1):
        x, y = X_2d[db_idx]

        class_txt = ""
        if "class_name" in index_df.columns:
            class_txt = f"\n{index_df.iloc[db_idx]['class_name']}"

        plt.scatter(
            x, y,
            s=180,
            c=[sims[db_idx]],
            cmap=cmap,
            norm=norm,
            edgecolors="black",
            linewidths=1.4,
            zorder=3,
        )
        plt.text(
            x, y,
            f"Top {rank_i}{class_txt}",
            fontsize=9,
            fontweight="bold",
            zorder=4,
        )

    qx, qy = X_2d[query_idx]
    plt.scatter(
        qx, qy,
        s=300,
        marker="*",
        c="red",
        edgecolors="black",
        linewidths=1.4,
        label="Query",
        zorder=5,
    )
    plt.text(qx, qy, "QUERY", fontsize=10, fontweight="bold", zorder=6)

    plt.title(
        f"PCA of 2D image embeddings colored by cosine similarity\n"
        f"PC1: {explained[0]*100:.2f}% variance, PC2: {explained[1]*100:.2f}% variance"
    )
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_saved_query_pca(
    model_dir: str,
    q_idx: int,
    top_k: int = 5,
):
    query_emb, results_for_query, query_path, test_index_df = load_saved_query_and_results(
        model_dir=model_dir,
        q_idx=q_idx,
    )

    show_pca_with_similarity_coloring(
        query_emb=query_emb,
        results=results_for_query,
        index_csv=os.path.join(model_dir, "train_index.csv"),
        embed_npy=os.path.join(model_dir, "train_embeddings.npy"),
        query_path=query_path,
        top_k=top_k,
    )

    print(f"Query index: {q_idx}")
    print(f"Query path: {query_path}")
    if "class_name" in test_index_df.columns:
        print(f"Query class: {test_index_df.iloc[q_idx]['class_name']}")

model_dir = os.path.join(RESULTS_DIR, "medsiglip")
plot_saved_query_pca(model_dir=model_dir, q_idx=0, top_k=5)

In [ ]:
!zip -r retrieval_outputs_2d.zip retrieval_outputs_2d/

In [ ]:
import os
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd


def load_saved_query_and_results(model_dir: str, q_idx: int) -> Tuple[np.ndarray, List[dict], str, pd.DataFrame]:
    test_index_csv = os.path.join(model_dir, "test_index.csv")
    test_embed_npy = os.path.join(model_dir, "test_embeddings.npy")
    retrieval_csv = os.path.join(model_dir, "retrieval_examples.csv")

    for path in (test_index_csv, test_embed_npy, retrieval_csv):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Missing file: {path}")

    test_index_df = pd.read_csv(test_index_csv)
    test_embeddings = np.load(test_embed_npy).astype(np.float32)
    retrieval_df = pd.read_csv(retrieval_csv)

    if q_idx < 0 or q_idx >= len(test_index_df):
        raise IndexError(f"q_idx={q_idx} out of range")

    query_path = test_index_df.iloc[q_idx]["image_path"]
    query_emb = test_embeddings[q_idx]

    results_for_query = (
        retrieval_df[retrieval_df["query_path"] == query_path]
        .sort_values("rank")
        .to_dict(orient="records")
    )

    if len(results_for_query) == 0:
        raise ValueError(f"No retrieval results found for query_path: {query_path}")

    return query_emb, results_for_query, query_path, test_index_df


def _extract_retrieved_paths(results: List[dict], top_k: Optional[int] = None) -> List[str]:
    if top_k is not None:
        results = results[:top_k]

    retrieved_paths = []
    for item in results:
        if "image_path" in item:
            retrieved_paths.append(item["image_path"])
        elif "retrieved_path" in item:
            retrieved_paths.append(item["retrieved_path"])
        else:
            raise KeyError("Missing image path key")
    return retrieved_paths


def _build_class_colors(class_names: List[str], cmap_name: str = "tab20") -> dict:
    import matplotlib.pyplot as plt

    unique_classes = sorted(pd.unique(class_names))
    cmap = plt.get_cmap(cmap_name, max(len(unique_classes), 1))
    return {cls_name: cmap(i) for i, cls_name in enumerate(unique_classes)}


def show_pca_with_class_coloring(
    query_emb: np.ndarray,
    results: List[dict],
    index_csv: str,
    embed_npy: str,
    query_path: Optional[str] = None,
    top_k: Optional[int] = None,
):
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA

    if not os.path.isfile(index_csv) or not os.path.isfile(embed_npy):
        raise FileNotFoundError("Missing index or embedding file")

    index_df = pd.read_csv(index_csv)
    db_embeddings = np.load(embed_npy).astype(np.float32)

    if "class_name" not in index_df.columns:
        raise KeyError("index_csv must contain 'class_name'")

    # Normalize
    db_embeddings = db_embeddings / (np.linalg.norm(db_embeddings, axis=1, keepdims=True) + 1e-8)
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-8)

    # Retrieved indices
    retrieved_paths = _extract_retrieved_paths(results, top_k=top_k)
    db_paths_abs = [os.path.abspath(p) for p in index_df["image_path"].tolist()]
    path_to_idx = {p: i for i, p in enumerate(db_paths_abs)}

    retrieved_indices = [
        path_to_idx[os.path.abspath(p)]
        for p in retrieved_paths
        if os.path.abspath(p) in path_to_idx
    ]

    # PCA
    X = np.vstack([db_embeddings, query_emb[None, :]])
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)
    explained = pca.explained_variance_ratio_
    query_idx = len(X) - 1

    # Colors
    class_names = index_df["class_name"].astype(str).tolist()
    class_to_color = _build_class_colors(class_names)

    plt.figure(figsize=(10, 8))

    # Plot all points (smaller size)
    for cls_name in sorted(set(class_names)):
        mask = index_df["class_name"].astype(str).values == cls_name
        plt.scatter(
            X_2d[:-1][mask, 0],
            X_2d[:-1][mask, 1],
            s=18,  # smaller points
            alpha=0.6,
            color=class_to_color[cls_name],
            label=cls_name,
        )

    # Highlight retrieved
    for rank_i, db_idx in enumerate(retrieved_indices, start=1):
        x, y = X_2d[db_idx]
        cls_name = str(index_df.iloc[db_idx]["class_name"])

        plt.scatter(
            x,
            y,
            s=140,
            color=class_to_color[cls_name],
            edgecolors="black",
            linewidths=1.5,
            zorder=4,
        )

        # Small clean labels (ONLY top-3)
        if rank_i <= 3:
            plt.text(
                x,
                y,
                f"{rank_i}",
                fontsize=7,
                ha="center",
                va="center",
                zorder=5,
                bbox=dict(
                    facecolor="white",
                    alpha=0.7,
                    edgecolor="none",
                    boxstyle="round,pad=0.2",
                ),
            )

    # Query point
    qx, qy = X_2d[query_idx]
    plt.scatter(
        qx,
        qy,
        s=250,
        marker="*",
        color="red",
        edgecolors="black",
        linewidths=1.5,
        label="Query",
        zorder=6,
    )

    plt.title(
        f"PCA of embeddings (colored by class)\n"
        f"PC1: {explained[0]*100:.2f}%, PC2: {explained[1]*100:.2f}%"
    )
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, alpha=0.3)

    # Clean legend (avoid clutter)
    if len(set(class_names)) <= 15:
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()


def plot_saved_query_pca_by_class(model_dir: str, q_idx: int, top_k: int = 5):
    query_emb, results, query_path, test_index_df = load_saved_query_and_results(
        model_dir=model_dir,
        q_idx=q_idx,
    )

    show_pca_with_class_coloring(
        query_emb=query_emb,
        results=results,
        index_csv=os.path.join(model_dir, "train_index.csv"),
        embed_npy=os.path.join(model_dir, "train_embeddings.npy"),
        query_path=query_path,
        top_k=top_k,
    )

    print(f"Query index: {q_idx}")
    print(f"Query path: {query_path}")
    if "class_name" in test_index_df.columns:
        print(f"Query class: {test_index_df.iloc[q_idx]['class_name']}")


# Usage
model_dir = os.path.join(RESULTS_DIR, "uni")
plot_saved_query_pca_by_class(model_dir=model_dir, q_idx=0, top_k=5)

In [ ]:
!cp -r retrieval_outputs_2d.zip /content/drive/MyDrive/Lung_Colon/